# 05 — Behavioral Friction (LSTM Model)

**Purpose:** Build an LSTM-based model to detect **behavioral friction** —
when a student becomes "stuck" on a knowledge node based on their clickstream
sequence patterns.

## Theory

We model P(stuck) from per-session clickstream features:

$$P(\text{stuck}_t) = \sigma\left(\text{LSTM}\left([\text{view\_count},\ \text{backtrack\_ratio},\ \text{avg\_dwell},\ \text{incorrect\_ratio}]\right)\right)$$

A student is **stuck** when they repeatedly revisit content and struggle with
assessments — a signal to trigger an AI recommendation.

## Architecture

```
Session features (T, 4)  -->  LSTM (hidden=64)  -->  FC(1)  -->  Sigmoid --> P(stuck)
```

## Feature Definitions

| Feature | Description |
|---------|-------------|
| `view_count` | Number of content view events in session |
| `backtrack_ratio` | Fraction of events that revisit previously seen content |
| `avg_dwell_time_proxy` | Mean time gap between consecutive events (seconds) |
| `incorrect_ratio` | Fraction of quick_check events answered incorrectly |

## Label

`stuck = 1` if `incorrect_ratio > 0.6 AND view_count > 2`

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
DUCKDB_PATH = os.environ.get('BDC_DUCKDB_PATH', '../data/student_analytics.duckdb')

print('Environment ready.')

## Section A: Feature Engineering

In [ ]:
# --- Load interaction data ---
df_interactions = None

if os.path.exists(DUCKDB_PATH):
    try:
        from scripts.load_data import load_duckdb_view
        df_interactions = load_duckdb_view(
            "SELECT user_id, node_id, action_type, is_correct, created_at "
            "FROM unified_interactions ORDER BY user_id, node_id, created_at"
        )
        print(f'[DuckDB] Loaded {len(df_interactions):,} interactions.')
    except Exception as e:
        print(f'[DuckDB] Failed: {e}')

if df_interactions is None:
    parquet_path = os.path.join(GOLD_DIR, 'gold_user_item_matrix.parquet')
    if os.path.exists(parquet_path):
        df_interactions = pd.read_parquet(parquet_path)
        rng = np.random.default_rng(0)
        if 'action_type' not in df_interactions.columns:
            df_interactions['action_type'] = rng.choice(['view', 'quick_check', 'learn', 'review'],
                                                        len(df_interactions))
        if 'is_correct' not in df_interactions.columns:
            df_interactions['is_correct'] = rng.integers(0, 2, len(df_interactions))
        if 'created_at' not in df_interactions.columns:
            df_interactions['created_at'] = pd.date_range('2024-01-01', periods=len(df_interactions), freq='5min')
        print(f'[Parquet] Loaded {len(df_interactions):,} interactions.')

if df_interactions is None:
    print('[Demo] Generating synthetic clickstream data...')
    rng = np.random.default_rng(42)
    n = 8000
    base_time = pd.Timestamp('2024-01-01')
    df_interactions = pd.DataFrame({
        'user_id':    rng.integers(1, 51, n),
        'node_id':    rng.integers(100, 130, n),
        'action_type': rng.choice(['view', 'quick_check', 'learn', 'review'],
                                  n, p=[0.4, 0.3, 0.2, 0.1]),
        'is_correct': rng.integers(0, 2, n),
        'created_at': [base_time + pd.Timedelta(seconds=int(x))
                       for x in rng.integers(0, 86400 * 30, n)],
    })

print(f'Interaction data: {df_interactions.shape}')
display(df_interactions.head(5))

In [ ]:
# --- Compute per-session (user, node) features ---
df_sorted = df_interactions.sort_values(['user_id', 'node_id', 'created_at']).copy()

def compute_session_features(group):
    """Compute behavioral friction features for a (user, node) session."""
    total_events = len(group)

    # view_count: number of 'view' actions
    view_count = (group['action_type'] == 'view').sum()

    # backtrack_ratio: revisit same node_id after leaving (proxy: view actions / total)
    backtrack_ratio = view_count / total_events if total_events > 0 else 0.0

    # avg_dwell_time_proxy: mean time gap between consecutive events
    if 'created_at' in group.columns and len(group) > 1:
        times = pd.to_datetime(group['created_at']).sort_values()
        diffs = times.diff().dt.total_seconds().dropna()
        avg_dwell = diffs.clip(0, 3600).mean()  # cap at 1 hour
    else:
        avg_dwell = 0.0

    # incorrect_ratio: among quick_check events, fraction incorrect
    qc = group[group['action_type'] == 'quick_check']
    if len(qc) > 0 and 'is_correct' in qc.columns:
        incorrect_ratio = 1.0 - qc['is_correct'].mean()
    else:
        incorrect_ratio = 0.0

    # Stuck label: heuristic rule
    stuck = int(incorrect_ratio > 0.6 and view_count > 2)

    return pd.Series({
        'view_count': float(view_count),
        'backtrack_ratio': float(backtrack_ratio),
        'avg_dwell_time_proxy': float(avg_dwell),
        'incorrect_ratio': float(incorrect_ratio),
        'total_events': float(total_events),
        'stuck': stuck,
    })

print('Computing session features...')
session_features = df_sorted.groupby(['user_id', 'node_id']).apply(compute_session_features).reset_index()

print(f'Session feature matrix: {session_features.shape}')
print(f'Stuck sessions: {session_features["stuck"].sum():,} / {len(session_features):,} '
      f'({session_features["stuck"].mean():.1%})')
display(session_features.head(10))

## Section B: LSTM Model Definition

In [ ]:
import torch
import torch.nn as nn

class FrictionLSTM(nn.Module):
    """
    LSTM-based model for predicting P(stuck) from behavioral session features.

    Input:  (batch, seq_len, input_dim)  where input_dim = 4 features
    Output: (batch, 1)  — P(stuck) in [0, 1] via sigmoid
    """

    def __init__(self, input_dim: int = 4, hidden_dim: int = 64,
                 num_layers: int = 1, dropout: float = 0.2):
        super(FrictionLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : torch.Tensor of shape (batch, seq_len, input_dim)

        Returns
        -------
        prob : torch.Tensor of shape (batch, 1) — P(stuck)
        """
        out, (h_n, _) = self.lstm(x)
        # Use last hidden state
        last_hidden = h_n[-1]          # (batch, hidden_dim)
        last_hidden = self.dropout(last_hidden)
        logit = self.fc(last_hidden)   # (batch, 1)
        return torch.sigmoid(logit)


model = FrictionLSTM(input_dim=4, hidden_dim=64, num_layers=1)
print('FrictionLSTM architecture:')
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

## Section C: Training Loop

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

FEATURE_COLS = ['view_count', 'backtrack_ratio', 'avg_dwell_time_proxy', 'incorrect_ratio']

# Normalize features
X = session_features[FEATURE_COLS].fillna(0.0).values.astype(np.float32)
y = session_features['stuck'].values.astype(np.float32)

# Normalize each feature to [0, 1]
X_min = X.min(axis=0, keepdims=True)
X_max = X.max(axis=0, keepdims=True) + 1e-9
X_norm = (X - X_min) / (X_max - X_min)

if len(X_norm) < 10:
    print('[Demo] Using small synthetic dataset for training demo...')
    rng = np.random.default_rng(99)
    X_norm = rng.random((200, 4)).astype(np.float32)
    y = (rng.random(200) > 0.7).astype(np.float32)

# Reshape to (N, seq_len=1, features=4) for LSTM
X_tensor = torch.from_numpy(X_norm).unsqueeze(1)  # (N, 1, 4)
y_tensor  = torch.from_numpy(y).unsqueeze(1)       # (N, 1)

dataset    = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion  = nn.BCELoss()
optimizer  = optim.Adam(model.parameters(), lr=1e-3)

N_EPOCHS = 5
epoch_losses = []

model.train()
for epoch in range(1, N_EPOCHS + 1):
    total_loss = 0.0
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    epoch_losses.append(avg_loss)
    print(f'Epoch {epoch}/{N_EPOCHS} | Loss: {avg_loss:.4f}')

print('\nTraining complete.')

## Section D: Inference — Predict P(stuck)

In [ ]:
model.eval()

def predict_stuck_probability(user_session_features: dict) -> float:
    """
    Predict P(stuck) for a user-concept session.

    Parameters
    ----------
    user_session_features : dict with keys:
        view_count, backtrack_ratio, avg_dwell_time_proxy, incorrect_ratio

    Returns
    -------
    float in [0, 1] — probability that the student is stuck
    """
    feat = np.array([
        user_session_features.get('view_count', 0),
        user_session_features.get('backtrack_ratio', 0),
        user_session_features.get('avg_dwell_time_proxy', 0),
        user_session_features.get('incorrect_ratio', 0),
    ], dtype=np.float32)

    # Normalize using training stats
    feat_norm = (feat - X_min[0]) / (X_max[0] - X_min[0])
    x_tensor = torch.from_numpy(feat_norm).unsqueeze(0).unsqueeze(0)  # (1, 1, 4)

    with torch.no_grad():
        prob = model(x_tensor).item()
    return prob


# Example inference calls
print('=== Inference Examples ===')

examples = [
    {'user': 'Student A (likely stuck)',
     'features': {'view_count': 5, 'backtrack_ratio': 0.7, 'avg_dwell_time_proxy': 120, 'incorrect_ratio': 0.8}},
    {'user': 'Student B (probably fine)',
     'features': {'view_count': 1, 'backtrack_ratio': 0.1, 'avg_dwell_time_proxy': 30, 'incorrect_ratio': 0.1}},
    {'user': 'Student C (borderline)',
     'features': {'view_count': 3, 'backtrack_ratio': 0.4, 'avg_dwell_time_proxy': 60, 'incorrect_ratio': 0.55}},
]

for ex in examples:
    p_stuck = predict_stuck_probability(ex['features'])
    label = '🔴 STUCK' if p_stuck > 0.5 else '🟢 OK'
    print(f'  {ex["user"]:<35} -> P(stuck) = {p_stuck:.4f}  {label}')

print('\nNote: Trigger AI recommendation when P(stuck) > 0.5')